In [ ]:
import sys
sys.setrecursionlimit(30000)

from Bio import Phylo
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import glob, os, re

TREE_DIR = "/data/users/ltucker/influenzaData/H5N1_pipeline/output/iqtree"
FASTA_DIR = "/data/users/ltucker/influenzaData/H5N1_pipeline/output/segments"
EMBED_DIR = "/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_per_segment_analysis/embeddings_tuned"

SEGMENT_TO_GENE = {
    "1": "PB2", "2": "PB1", "3": "PA", "4": "HA",
    "5": "NP",  "6": "NA",  "7": "MP", "8": "NS",
}

CONTINENT_COLORS = {
    "africa": "#e6194b",
    "asia": "#3cb44b",
    "europe": "#4363d8",
    "americas": "#f58231",
    "oceania": "#911eb4",
    "north america": "#f58231",
    "south america": "#42d4f4",
}

In [ ]:
def extract_epi(header):
    m = re.search(r'EPI_ISL_\d+', header)
    return m.group(0) if m else None


def load_original_headers(segment_num):
    path = os.path.join(FASTA_DIR, f"{segment_num}.fa")
    headers = {}
    with open(path) as f:
        for line in f:
            if line.startswith(">"):
                h = line.strip().lstrip(">")
                epi = extract_epi(h)
                if epi:
                    headers[epi] = h
    return headers


def load_embeddings(segment_num):
    gene = SEGMENT_TO_GENE[str(segment_num)]
    path = os.path.join(EMBED_DIR, f"embeddings_{gene}.parquet")
    emb = pd.read_parquet(path)
    emb["epi_id"] = emb["sample_id"].apply(extract_epi)
    return emb


def parse_metadata(df):
    """Parse country, continent, year from original_header."""
    split = df["original_header"].str.split("|")
    df = df.copy()
    df["country"] = split.str[5].str.strip().str.lower()
    df["continent"] = split.str[6].str.strip().str.lower()
    df["year"] = pd.to_numeric(split.str[7].str.strip(), errors="coerce").astype("Int64")
    return df


def build_segment_df(segment_num):
    """Build full DataFrame: iqtree header, original header, embeddings, metadata."""
    tree = Phylo.read(os.path.join(TREE_DIR, f"segment_{segment_num}.treefile"), "newick")
    orig_headers = load_original_headers(segment_num)
    emb = load_embeddings(segment_num)

    rows = []
    for tip in tree.get_terminals():
        epi = extract_epi(tip.name)
        rows.append({
            "epi_id": epi,
            "iqtree_header": tip.name,
            "original_header": orig_headers.get(epi, "NOT_FOUND"),
        })
    df = pd.DataFrame(rows)

    # Merge embeddings on epi_id
    df = df.merge(emb, on="epi_id", how="left")
    df.drop(columns=["sample_id"], inplace=True, errors="ignore")

    # Parse metadata
    df = parse_metadata(df)

    gene = SEGMENT_TO_GENE[str(segment_num)]
    df.insert(0, "segment", f"segment_{segment_num}")
    df.insert(1, "gene", gene)

    n_matched = df["tsne_1"].notna().sum()
    print(f"  segment_{segment_num} ({gene}): {len(df)} tips, {n_matched} embeddings matched")
    return df, tree


# Build all segments
all_dfs = {}
all_trees = {}
for seg_num in sorted(SEGMENT_TO_GENE.keys(), key=int):
    all_dfs[seg_num], all_trees[seg_num] = build_segment_df(seg_num)

print(f"\nDone — {len(all_dfs)} segments loaded")

In [ ]:
seg = "1"  # change to inspect other segments
df = all_dfs[seg]
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Years: {sorted(df['year'].dropna().unique())}")
print(f"Continents: {sorted(df['continent'].dropna().unique())}")
pd.set_option('display.max_colwidth', None)
df["iqtree_header"].head(10)

In [ ]:
def plot_segment_by_continent(seg_num):
    df = all_dfs[seg_num]
    tree = all_trees[seg_num]
    gene = SEGMENT_TO_GENE[seg_num]

    # Colour tips by continent
    epi_to_continent = dict(zip(df["epi_id"], df["continent"]))
    for tip in tree.get_terminals():
        epi = extract_epi(tip.name)
        continent = epi_to_continent.get(epi, "unknown")
        tip.color = CONTINENT_COLORS.get(continent, "#cccccc")
    for clade in tree.find_clades(terminal=False):
        clade.color = "#cccccc"

    # Layout: tree on top (full width), 4 embeddings below
    fig = plt.figure(figsize=(24, 14))
    gs = fig.add_gridspec(2, 4, height_ratios=[2, 1], hspace=0.3)

    # Tree spans all 4 columns on top row
    ax_tree = fig.add_subplot(gs[0, :])
    Phylo.draw(tree, axes=ax_tree, do_show=False, label_func=lambda x: "")
    ax_tree.set_title(f"Phylogeny — segment_{seg_num} ({gene})", fontsize=20)

    # 4 embedding plots on bottom row
    methods = [("tsne_1", "tsne_2", "t-SNE"),
               ("umap_1", "umap_2", "UMAP"),
               ("mds_1", "mds_2", "MDS"),
               ("phate_1", "phate_2", "PHATE")]

    for col, (x, y, title) in enumerate(methods):
        ax = fig.add_subplot(gs[1, col])
        for continent in sorted(df["continent"].dropna().unique()):
            sub = df[df["continent"] == continent]
            color = CONTINENT_COLORS.get(continent, "#cccccc")
            ax.scatter(sub[x], sub[y], c=color, s=3, alpha=0.5, label=continent)
        ax.set_title(title, fontsize=20)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    # Single legend
    handles, labels = fig.axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(labels),
           fontsize=15, markerscale=3, framealpha=0.9,
           bbox_to_anchor=(0.5, -0.02))

    plt.suptitle(f"Segment {seg_num} ({gene}) — coloured by continent", fontsize=30)
    plt.subplots_adjust(bottom=0.06)
    plt.show()

# Plot all segments
for seg_num in sorted(SEGMENT_TO_GENE.keys(), key=int):
    plot_segment_by_continent(seg_num)


In [ ]:
def plot_segment_by_year(seg_num):
    df = all_dfs[seg_num]
    tree = all_trees[seg_num]
    gene = SEGMENT_TO_GENE[seg_num]

    years = df["year"].dropna()
    if years.empty:
        print(f"  segment_{seg_num}: no year data, skipping")
        return
    ymin, ymax = int(years.min()), int(years.max())
    norm = mcolors.Normalize(vmin=ymin, vmax=ymax)
    cmap = cm.viridis

    # Colour tips by year
    epi_to_year = dict(zip(df["epi_id"], df["year"]))
    for tip in tree.get_terminals():
        epi = extract_epi(tip.name)
        yr = epi_to_year.get(epi)
        if pd.notna(yr):
            tip.color = mcolors.to_hex(cmap(norm(int(yr))))
        else:
            tip.color = "#cccccc"
    for clade in tree.find_clades(terminal=False):
        clade.color = "#cccccc"

    fig = plt.figure(figsize=(24, 14))
    gs = fig.add_gridspec(2, 4, height_ratios=[2, 1], hspace=0.3)

    # Tree on top
    ax_tree = fig.add_subplot(gs[0, :])
    Phylo.draw(tree, axes=ax_tree, do_show=False, label_func=lambda x: "")
    ax_tree.set_title(f"Phylogeny — segment_{seg_num} ({gene})", fontsize=20)

    # Embeddings below
    methods = [("tsne_1", "tsne_2", "t-SNE"),
               ("umap_1", "umap_2", "UMAP"),
               ("mds_1", "mds_2", "MDS"),
               ("phate_1", "phate_2", "PHATE")]

    for col, (x, y, title) in enumerate(methods):
        ax = fig.add_subplot(gs[1, col])
        valid = df.dropna(subset=[x, y, "year"])
        ax.scatter(valid[x], valid[y], c=valid["year"], cmap=cmap,
                   norm=norm, s=3, alpha=0.5)
        ax.set_title(title, fontsize=20)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

# Horizontal colorbar at bottom
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar_ax = fig.add_axes([0.25, 0.02, 0.5, 0.015])  # [left, bottom, width, height]
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
    cbar.set_label("Year", fontsize=20)

    plt.suptitle(f"Segment {seg_num} ({gene}) — coloured by year", fontsize=14)
    plt.subplots_adjust(bottom=0.08)
    plt.show()

# Plot all segments
for seg_num in sorted(SEGMENT_TO_GENE.keys(), key=int):
    plot_segment_by_year(seg_num)

In [ ]:
summary = pd.read_csv(f"{clust_dir}/clustering_summary.csv")
print(summary)